In [1]:
class Product:

  """
  This class represents a product
  """

  def __init__(self, product_name, qty, purchase_price, sale_price):

    """
    product_name(str) = name of the product
    qty(int) = quantity
    purchase_price(float) = purchase price
    sale_price(float) = sale price
    """

    self._product_name = product_name
    self._qty = int(qty)
    self._purchase_price = float(purchase_price)
    self._sale_price = float(sale_price)


  def add_to_list(self, product_list):

    """
    This function updates the quantity of a product in the list / adds the product
    """

    for product in product_list:
      if product._product_name==self._product_name:
        product._qty+=self._qty
        return

    product_list.append(self)


  def remove_from_list(self, product_list):

    """
    This function updates the quantity of a product in the list / removes the product
    """

    for product in product_list:
      if product._product_name==self._product_name:
        product._qty-=self._qty
        if product._qty<=0:
          product_list.remove(product)
        return


  def __repr__(self):
    info = f"{self._product_name}\t{self._qty}\t{self._sale_price}\n"
    return info

In [2]:
import csv

class Management_System:

  """
  This class represents a management system
  """


  def __init__(self):

    """
    stock(list) = products stock
    sales(list) = product sales
    """

    self._stock = []
    self._sales = []
    self.load_csv("sales.csv", self._sales)
    self.load_csv("stock.csv", self._stock)



  def load_csv(self, csv_file_name, product_list):

    """
    This function loads the information about sales and stock from a CSV file
    """

    try:
      with open(csv_file_name, "r", encoding="utf-8") as csv_file:
        csv_reader = csv.DictReader(csv_file)

        for item in csv_reader:
          product = Product(product_name = item["Name"],
                            qty = int(item["Quantity"]),
                            purchase_price = float(item["Purchase price"]),
                            sale_price = float(item["Sale price"])
                            )
          product_list.append(product)

    except FileNotFoundError:
      product_list = []

    return product_list



  def write_csv(self, csv_file_name, product_list):

    """
    This function overwrites the information about sales and stock on a CSV file
    """

    fieldnames = ["Name", "Quantity", "Purchase price", "Sale price"]   # We define the headers

    with open(csv_file_name, "w", newline="", encoding="utf-8") as csv_file:
      csv_writer = csv.DictWriter(csv_file, fieldnames=fieldnames)

      csv_writer.writeheader()      # We write the headers on file

      for item in product_list:
        csv_writer.writerow({
            "Name":item._product_name,
            "Quantity":item._qty,
            "Purchase price":item._purchase_price,
            "Sale price":item._sale_price
        })

    return



  def add_product(self):

    """
    This function registers the products that are input in stock
    """

    product_name = input("Nome del prodotto: ")

    try:
      qty = int(input("Quantità: "))
      assert qty > 0
    except AssertionError:
      print("Inserire un valore maggiore di zero.")
      qty = int(input("Quantità: "))
    except ValueError:
      print("Numero non valido.")
      qty = int(input("Quantità: "))

    is_found = False

    for item in self._stock:
      if item._product_name==product_name:
        item._qty+=qty
        is_found = True

    if is_found==False:

      try:
        purchase_price = float(input("Prezzo di acquisto: "))
        assert purchase_price > 0
      except AssertionError:
        print("Inserire un valore maggiore di zero.")
        purchase_price = float(input("Prezzo di acquisto: "))
      except ValueError:
        print("Numero non valido.")
        purchase_price = float(input("Prezzo di acquisto: "))

      try:
        sale_price = float(input("Prezzo di vendita: "))
        assert sale_price > 0
      except AssertionError:
        print("Inserire un valore maggiore di zero.")
        sale_price = float(input("Prezzo di vendita: "))
      except ValueError:
        print("Numero non valido.")
        sale_price = float(input("Prezzo di vendita: "))

      self._stock.append(Product(product_name, qty, purchase_price, sale_price))

    print(f"AGGIUNTO: {qty} X {product_name}")

    ms.write_csv("stock.csv", self._stock)       # We call the function that updates the information in stock

    return



  def show_stock(self):

    """
    This function lists the products in stock
    """

    print("PRODOTTO\tQUANTITA'\tPREZZO\t")

    for item in self._stock:
      print(f"{item._product_name}\t{item._qty}\t€{item._sale_price:.2f}\t")

    return



  def register_sale(self):

    """
    This function registers the sales
    """

    current_sale = []
    sale_amount = 0        # Sale amount initialised at zero


    add_product = "si"      # Sale of a product already initialised at zero

    while add_product=="si":

      product_name = input("Nome del prodotto: ")

      is_in_stock = False

      for item in self._stock:

        if item._product_name==product_name:   # We verify that the product is in stock

          try:
            qty = int(input("Quantità: "))
            assert qty > 0
          except AssertionError:
            print("Inserire un valore maggiore di zero.")
            qty = int(input("Quantità: "))
          except ValueError:
            print("Numero non valido.")
            qty = int(input("Quantità: "))

          is_in_stock = True

          if item._qty>=qty:      # We verify that the requested quantity is in stock

            is_in_sales = False


            current_product = Product(product_name, qty, item._purchase_price, item._sale_price)

            current_product.add_to_list(self._sales)    # We add the product to the list of all sold products
            current_product.add_to_list(current_sale)   # We add the product to the list of the current sale
            current_product.remove_from_list(self._stock)   # We remove the product from stock

            break

          else:
            print("La quantità richiesta non è presente in magazzino")

      if is_in_stock==False:
        print("Il prodotto non è presente in magazzino")

      add_product = input("Aggiungere un altro prodotto? (si/no): ")

      while add_product!="si" and add_product!="no":
        print("Risposta non valida!")
        add_product = input("Aggiungere un altro prodotto? (si/no): ")


    print("VENDITA REGISTRATA")

    for item in current_sale:
      sale_amount+=(item._qty*item._sale_price)
      print(f"- {item._qty} X {item._product_name}: €{item._sale_price:.2f}")

    print(f"Totale: €{sale_amount:.2f}")

    ms.write_csv("sales.csv", self._sales)     # We update the CSV file with all sales
    ms.write_csv("stock.csv", self._stock)     # We update the CSV file with the stock

    return



  def show_profits(self):

    """
    This function shows the gross and net profits
    """

    gross_profit = 0
    net_profit = 0

    for item in self._sales:
      gross_profit += item._qty*item._sale_price
      net_profit += item._qty*(item._sale_price-item._purchase_price)

    print(f"Profitto: lordo=€{gross_profit:.2f} netto=€{net_profit:.2f}")



  def show_commands(self):

    """
    This function shows the available commands
    """

    print("I comandi disponibili sono i seguenti: "+
        "\naggiungi: aggiungi un prodotto al magazzino"+
        "\nelenca: elenca i prodotti in magazzino"+
        "\nvendita: registra una vendita effettuata"+
        "\nprofitti: mostra i profitti totali"+
        "\naiuto: mostra i possibili comandi"+
        "\nchiudi: esci dal programma")

In [ ]:
ms = Management_System()
is_on = True

while is_on:
  command = input("Inserisci un comando: ")

  if command=="aggiungi":
    ms.add_product()
    print("\n")

  elif command=="elenca":
    ms.show_stock()
    print("\n")

  elif command=="vendita":
    ms.register_sale()
    print("\n")

  elif command=="profitti":
    ms.show_profits()
    print("\n")

  elif command=="aiuto":
    ms.show_commands()
    print("\n")

  elif command=="chiudi":
    print("Bye bye")
    is_on = False

  else:
    print("Comando non valido")
    ms.show_commands()
    print("\n")